# Milestone 5: Multimodal with posters

Starter notebook for milestone 5 of [Project - MovieLens Recommender](Project%20-%20MovieLens%20Recommender.md).

Cells marked **TODO** are yours to fill in; everything else runs as-is. Notebook is saved without outputs.

In [ ]:
import numpy as np, pandas as pd, pathlib, urllib.request, zipfile, io

DATA = pathlib.Path("ml-latest-small")
if not DATA.exists():
    # Or download manually from https://grouplens.org/datasets/movielens/ (ml-latest-small) and unzip here
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
ratings = pd.read_csv(DATA / "ratings.csv")   # userId, movieId, rating, timestamp
movies = pd.read_csv(DATA / "movies.csv")     # movieId, title, genres
links = pd.read_csv(DATA / "links.csv")       # movieId, imdbId, tmdbId
ratings.head()

In [ ]:
# Leave-last-out split by time: for every user, the most recent positive is the test item,
# the one before it is validation. Positives = rating >= 4 (implicit feedback).
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)   # 0 = most recent
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]

# Contiguous ids so we can index embedding tables later
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
iid = {m: i for i, m in enumerate(sorted(movies.movieId.unique()))}   # includes never-rated movies (cold items)
n_users, n_items = len(uid), len(iid)
print(n_users, n_items, len(train), len(val), len(test))

In [ ]:
def evaluate(score_fn, held_out, train, k=10, n_neg=99, seed=0):
    """HitRate@k and NDCG@k with 1 held-out positive vs n_neg sampled unseen items (sampled protocol).
    score_fn(user_id, item_ids) -> array of scores. Switch to full ranking for final numbers,
    sampled metrics can reorder models (see Evaluating Recommenders)."""
    rng = np.random.default_rng(seed)
    seen = train.groupby("userId").movieId.apply(set).to_dict()
    all_items = np.array(list(iid))
    hits, ndcg = [], []
    for u, m in held_out.itertuples(index=False):
        banned = seen.get(u, set()) | {m}
        negs = rng.choice(all_items, size=n_neg * 2, replace=False)
        negs = [x for x in negs if x not in banned][:n_neg]
        cand = np.array([m] + negs)
        s = np.asarray(score_fn(u, cand))
        rank = (s > s[0]).sum() + rng.integers(0, (s == s[0]).sum())   # 0 = first; ties broken at random (a constant scorer must score ~chance)
        hits.append(rank < k)
        ndcg.append(1 / np.log2(rank + 2) if rank < k else 0.0)
    return {"HR@k": float(np.mean(hits)), "NDCG@k": float(np.mean(ndcg))}

## 1. Fetch posters
MovieLens has no images, but `links.csv` maps `movieId` to a TMDB id. Create a free TMDB API key, fetch `poster_path` per movie (cache JSON to disk, be polite with rate limits), then download images at a small size (w185). Keep only movies where a poster exists and report how many that is.

In [ ]:
import os, json
TMDB_KEY = os.environ.get("TMDB_API_KEY")   # never commit keys
POSTERS = pathlib.Path("posters"); POSTERS.mkdir(exist_ok=True)
# TODO: GET https://api.themoviedb.org/3/movie/{tmdbId}?api_key=... -> "poster_path"
#       image URL: https://image.tmdb.org/t/p/w185{poster_path}
#       cache to posters/{movieId}.jpg and posters/index.json

## 2. Extract frozen features
Encode every poster with CLIP (image) and every title + genres string with the CLIP text encoder or Sentence-BERT. Save as `.npy`, never re-encode inside the training loop.

In [ ]:
# TODO (needs `pip install transformers torch pillow`):
# from transformers import CLIPModel, CLIPProcessor
# model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32"); proc = CLIPProcessor.from_pretrained(...)
# img_feat: (n_items_with_poster, 512) -> np.save("img_feat.npy", ...)

## 3. VBPR-style model
$\hat x_{ui}=\beta_i+\gamma_u^\top\gamma_i+\theta_u^\top(Ef_i)$ trained with BPR. Start from your M3 model and add the visual term with a learned projection $E$ on frozen CLIP features. Standardise features first.

In [ ]:
# TODO: class VBPR(nn.Module) ... reuse bpr_loss from M3

## 4. The cold-start test
Hide all training interactions of 10% of items (cold items). Evaluate only on test pairs whose item is cold. BPR-MF should be at chance, the multimodal model above it. This is the experiment that justifies the whole milestone.

In [ ]:
# TODO: cold-item split + evaluate both models; add a fusion ablation (image only / text only / both)